In [ ]:
# train_quiz_model.ipynb
# This would be a Jupyter notebook

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
import torch

# Load your dataset
df = pd.read_csv('question_dataset.csv')

# Preprocess data for training
def preprocess_data(df):
    # Create training pairs: (content, question)
    training_data = []
    
    for _, row in df.iterrows():
        # Use subject and year as context
        context = f"Subject: {row['Subject']}, Year: {row.get('Year', 'N/A')}"
        question = row['Question']
        
        training_data.append({
            'context': context,
            'question': question
        })
    
    return training_data

# Prepare dataset for model training
training_data = preprocess_data(df)

# Initialize model and tokenizer
model_name = "t5-small"  # Start with smaller model for fine-tuning
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Tokenize data
def tokenize_function(examples):
    inputs = [f"generate question: {ctx}" for ctx in examples['context']]
    targets = examples['question']
    
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=True)
    labels = tokenizer(targets, max_length=128, truncation=True, padding=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


print("Training data prepared!")
print(f"Total training examples: {len(training_data)}")